# 🎬 AGAR-RL V5 : Visionneuse HD du Dernier Checkpoint V5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Albin0903/agario/blob/main/notebooks/eval_drive_models.ipynb)

Ce notebook est spécialement configuré pour **Exécuter tout (Run All)** en 1 clic :
1. Connexion automatique à votre Google Drive (`agario_rl_backup_v5`).
2. Sélection automatique du **tout dernier checkpoint V5** (ex: `ppo_step_250000.zip` ou plus récent).
3. Génération et affichage direct de la **vidéo de gameplay HD** (80 secondes @ 30 FPS avec HUD, vecteurs et radar).


In [ ]:
# 1. Montage sécurisé de Google Drive
import os, sys
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except ImportError:
    pass

DRIVE_V5_DIR = '/content/drive/MyDrive/agario_rl_backup_v5'
print('=' * 65)
print('✅ Google Drive connecté.')
print(f'📁 Dossier cible V5 : {DRIVE_V5_DIR} (Existe: {os.path.exists(DRIVE_V5_DIR)})')
print('=' * 65)


In [ ]:
# 2. Synchronisation Git & Dépendances
import os

if os.path.exists('.git'):
    !git fetch origin main
    !git reset --hard origin/main
elif os.path.exists('agario/.git'):
    %cd agario
    !git fetch origin main
    !git reset --hard origin/main
else:
    !git clone https://github.com/Albin0903/agario.git
    %cd agario

os.environ['PYTHONPATH'] = f"{os.getcwd()}:{os.environ.get('PYTHONPATH', '')}"
!pip uninstall -y -q gym 2>/dev/null || true
!pip install -q -r requirements.txt
!apt-get install -qq -y ffmpeg
print('✅ Code et dépendances synchronisés avec succès.')


In [ ]:
# 3. Détection du Dernier Checkpoint V5 & Génération du Replay HD
import os, glob, re
from IPython.display import HTML, display
from base64 import b64encode

def extract_step(path):
    fname = os.path.basename(path)
    if 'final' in fname:
        return 999_999_999
    m = re.search(r'step_(\d+)', fname)
    return int(m.group(1)) if m else 0

v5_dir = '/content/drive/MyDrive/agario_rl_backup_v5'
candidates = glob.glob(os.path.join(v5_dir, '*.zip'))

# Fallback local si non exécuté sur Colab
if not candidates:
    for alt_dir in ['checkpoints/self_play_pool', 'checkpoints/ppo', '/content/drive/MyDrive/agario_rl_backup_v2']:
        if os.path.exists(alt_dir):
            candidates.extend(glob.glob(os.path.join(alt_dir, '*.zip')))

valid_candidates = [c for c in candidates if os.path.getsize(c) > 1000 and not os.path.basename(c).startswith('._')]
if not valid_candidates:
    raise FileNotFoundError('❌ Aucun checkpoint .zip trouvé sur Google Drive dans agario_rl_backup_v5. Vérifiez votre Drive.')

valid_candidates.sort(key=extract_step, reverse=True)
LATEST_MODEL = valid_candidates[0]
step_number = extract_step(LATEST_MODEL)

print('=' * 75)
print(f'🏆 DERNIER CHECKPOINT V5 DÉTECTÉ : {os.path.basename(LATEST_MODEL)}')
print(f'📊 Palier : {step_number:,} steps' if step_number < 999_999_999 else '📊 Palier : FINAL')
print(f'📁 Emplacement : {LATEST_MODEL}')
print('=' * 75)

# Génération de la vidéo HD (2400 steps = 80 secondes @ 30 FPS)
os.makedirs('recordings', exist_ok=True)
video_path = 'recordings/gameplay_v5_latest.mp4'

print('\n🎬 Enregistrement du match en cours (80s avec physique V5 exacte et 20 bots)...\n')
!python src/inference/record_match.py \
    --model "{LATEST_MODEL}" \
    --output "{video_path}" \
    --steps 2400 \
    --fps 30

# Copie miroir sur Google Drive
if os.path.exists(video_path) and os.path.exists(v5_dir):
    drive_copy = os.path.join(v5_dir, f'gameplay_v5_step_{step_number}.mp4')
    !cp "{video_path}" "{drive_copy}"
    print(f'📁 Vidéo également archivée sur votre Drive : {drive_copy}')

# Affichage direct dans le Notebook
if os.path.exists(video_path):
    mp4_bytes = open(video_path, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
    display(HTML(f'''
    <div style="text-align: center; margin: 25px 0;">
        <h2 style="color: #2c3e50; margin-bottom: 6px;">🎮 Gameplay HD - Dernier Checkpoint V5 ({step_number:,} steps)</h2>
        <p style="color: #7f8c8d; font-size: 14px; margin-bottom: 16px;">HUD dynamique, overlay des vecteurs de décision de l'IA et radar minimap</p>
        <video width="920" height="520" controls autoplay loop style="border-radius: 10px; box-shadow: 0 6px 20px rgba(0,0,0,0.35);">
            <source src="{data_url}" type="video/mp4">
        </video>
        <p style="color: #888; font-size: 13px; margin-top: 10px;">
            Fichier : <code>gameplay_v5_latest.mp4</code> ({os.path.getsize(video_path) / 1_000_000:.1f} Mo)
        </p>
    </div>
    '''))
else:
    print('⚠️ Erreur : La vidéo n\'a pas pu être générée.')
